In [1]:
# =========================================================
# TASK 18 — CELL 1
# SETUP + REPRODUCIBILITY
# =========================================================

import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold

# ---------------------------------------------------------
# PROJECT ROOT
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# REPRODUCIBILITY
# ---------------------------------------------------------

RANDOM_STATE = 42
N_SPLITS = 5

np.random.seed(RANDOM_STATE)

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# ---------------------------------------------------------
# DIRECTORIES
# ---------------------------------------------------------

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models"

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# VERIFICATION
# ---------------------------------------------------------

print("=" * 65)
print("        TASK 18 — NATURAL LANGUAGE PROCESSING")
print("=" * 65)

print("\nRandom state :", RANDOM_STATE)
print("CV strategy  : StratifiedKFold")
print("CV folds     :", N_SPLITS)

print("\nArtifacts directory:")
print(ARTIFACTS_DIR)

print("\nModels directory:")
print(MODELS_DIR)

print("\nSetup completed successfully.")

        TASK 18 — NATURAL LANGUAGE PROCESSING

Random state : 42
CV strategy  : StratifiedKFold
CV folds     : 5

Artifacts directory:
/home/akash/Projects/Altrodav/artifacts

Models directory:
/home/akash/Projects/Altrodav/models

Setup completed successfully.


In [2]:
# =========================================================
# TASK 18 — CELL 2
# LOAD REAL NLP DATASET
# =========================================================

from sklearn.datasets import fetch_20newsgroups

# ---------------------------------------------------------
# Load training and test text data
# ---------------------------------------------------------

train_data = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

test_data = fetch_20newsgroups(
    subset="test",
    remove=("headers", "footers", "quotes")
)

# ---------------------------------------------------------
# Create DataFrames
# ---------------------------------------------------------

train_df = pd.DataFrame({
    "text": train_data.data,
    "target": train_data.target
})

test_df = pd.DataFrame({
    "text": test_data.data,
    "target": test_data.target
})

# ---------------------------------------------------------
# Target names
# ---------------------------------------------------------

target_names = train_data.target_names

# ---------------------------------------------------------
# Dataset information
# ---------------------------------------------------------

print("========== NLP DATASET ==========")

print(
    "Training samples:",
    len(train_df)
)

print(
    "Test samples:",
    len(test_df)
)

print(
    "Number of classes:",
    len(target_names)
)

print("\n========== CLASSES ==========")

for index, name in enumerate(target_names):
    print(
        f"{index}: {name}"
    )

# ---------------------------------------------------------
# Dataset preview
# ---------------------------------------------------------

print("\n========== SAMPLE TEXT ==========")

print(
    train_df.iloc[0]["text"][:1000]
)

print("\n========== DATASET LOADING ==========")

print(
    "Training dataset loaded successfully."
)

print(
    "Independent test dataset loaded successfully."
)

========== NLP DATASET ==========
Training samples: 11314
Test samples: 7532
Number of classes: 20

========== CLASSES ==========
0: alt.atheism
1: comp.graphics
2: comp.os.ms-windows.misc
3: comp.sys.ibm.pc.hardware
4: comp.sys.mac.hardware
5: comp.windows.x
6: misc.forsale
7: rec.autos
8: rec.motorcycles
9: rec.sport.baseball
10: rec.sport.hockey
11: sci.crypt
12: sci.electronics
13: sci.med
14: sci.space
15: soc.religion.christian
16: talk.politics.guns
17: talk.politics.mideast
18: talk.politics.misc
19: talk.religion.misc

========== SAMPLE TEXT ==========
I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
hav

In [3]:
# =========================================================
# TASK 18 — CELL 3
# TEXT CLEANING + TOKENISATION
# =========================================================

def clean_and_tokenize(text):
    """
    Clean and tokenize a text document.

    Steps:
    1. Convert to lowercase
    2. Remove URLs
    3. Remove email addresses
    4. Keep alphabetic characters
    5. Normalize whitespace
    6. Tokenize
    7. Remove very short tokens
    """

    # Safety check
    if not isinstance(text, str):
        return []

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Keep alphabetic characters
    text = re.sub(
        r"[^a-z\s]",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenize
    tokens = text.split()

    # Remove very short tokens
    tokens = [
        token
        for token in tokens
        if len(token) > 2
    ]

    return tokens


def clean_text(text):
    """
    Return cleaned text as a single string.
    """

    return " ".join(
        clean_and_tokenize(text)
    )


# ---------------------------------------------------------
# Test cleaning on one document
# ---------------------------------------------------------

sample_raw_text = train_df.iloc[0]["text"]

sample_tokens = clean_and_tokenize(
    sample_raw_text
)

sample_clean_text = clean_text(
    sample_raw_text
)

# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print("========== TEXT CLEANING TEST ==========")

print("\nOriginal text:")
print(sample_raw_text[:500])

print("\nCleaned text:")
print(sample_clean_text[:500])

print("\nTokens:")
print(sample_tokens[:30])

print(
    "\nOriginal character count:",
    len(sample_raw_text)
)

print(
    "Cleaned character count:",
    len(sample_clean_text)
)

print(
    "Number of tokens:",
    len(sample_tokens)
)

print(
    "\nText cleaning and tokenisation completed successfully."
)

========== TEXT CLEANING TEST ==========

Original text:
I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.

Cleaned text:
was wondering anyone out there could enlighten this car saw the other day was door sports car looked from the late early was called bricklin the doors were really small addition the front bumper was separate from the rest the body this all know anyone can tellme model name engine specs years production where this car made history whatever info you have this funky looking car please mail

Tokens:
['was', 'wondering', 'anyone', 'out', 'there', 'could

In [4]:
# =========================================================
# TASK 18 — CELL 4
# TF-IDF VECTORIZATION
# =========================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------------------------------------------------
# Clean training and test text
# ---------------------------------------------------------

print("Cleaning training text...")

X_train_text = train_df["text"].apply(
    clean_text
)

print("Cleaning test text...")

X_test_text = test_df["text"].apply(
    clean_text
)

# ---------------------------------------------------------
# Target variables
# ---------------------------------------------------------

y_train = train_df["target"].values
y_test = test_df["target"].values

# ---------------------------------------------------------
# TF-IDF Vectorizer
# ---------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    max_features=50000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# ---------------------------------------------------------
# Fit ONLY on training data
# ---------------------------------------------------------

X_train_tfidf = tfidf_vectorizer.fit_transform(
    X_train_text
)

# ---------------------------------------------------------
# Transform test data
# ---------------------------------------------------------

X_test_tfidf = tfidf_vectorizer.transform(
    X_test_text
)

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("\n========== TF-IDF VECTORIZATION ==========")

print(
    "Training documents:",
    X_train_tfidf.shape[0]
)

print(
    "Test documents:",
    X_test_tfidf.shape[0]
)

print(
    "TF-IDF features:",
    X_train_tfidf.shape[1]
)

print(
    "Training matrix shape:",
    X_train_tfidf.shape
)

print(
    "Test matrix shape:",
    X_test_tfidf.shape
)

print(
    "\nVocabulary size:",
    len(tfidf_vectorizer.vocabulary_)
)

print(
    "\nN-gram range:",
    tfidf_vectorizer.ngram_range
)

print(
    "Maximum features:",
    tfidf_vectorizer.max_features
)

print(
    "\nTest leakage check:"
)

print(
    "Vectorizer fitted on training data only: PASS"
)

print(
    "\nTF-IDF vectorization completed successfully."
)

Cleaning training text...
Cleaning test text...

========== TF-IDF VECTORIZATION ==========
Training documents: 11314
Test documents: 7532
TF-IDF features: 50000
Training matrix shape: (11314, 50000)
Test matrix shape: (7532, 50000)

Vocabulary size: 50000

N-gram range: (1, 2)
Maximum features: 50000

Test leakage check:
Vectorizer fitted on training data only: PASS

TF-IDF vectorization completed successfully.


In [5]:
# =========================================================
# TASK 18 — CELL 5
# BASELINE NLP CLASSIFIER
# =========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# ---------------------------------------------------------
# Create baseline model
# ---------------------------------------------------------

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

# ---------------------------------------------------------
# Cross-validation
# ---------------------------------------------------------

print("========== BASELINE NLP MODEL ==========")

print("Model: Logistic Regression")
print("CV strategy: StratifiedKFold")
print("Number of folds:", N_SPLITS)

cv_scores = cross_val_score(
    baseline_model,
    X_train_tfidf,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

# ---------------------------------------------------------
# CV statistics
# ---------------------------------------------------------

cv_mean = cv_scores.mean()
cv_std = cv_scores.std()
cv_variance = cv_scores.var()

print("\n========== CROSS-VALIDATION ==========")

print(
    "Fold scores:",
    np.round(cv_scores, 4)
)

print(
    f"Mean CV Accuracy : {cv_mean:.4f}"
)

print(
    f"CV Std           : {cv_std:.4f}"
)

print(
    f"CV Variance      : {cv_variance:.4f}"
)

# ---------------------------------------------------------
# Train baseline model
# ---------------------------------------------------------

baseline_model.fit(
    X_train_tfidf,
    y_train
)

# ---------------------------------------------------------
# Independent test evaluation
# ---------------------------------------------------------

baseline_test_accuracy = baseline_model.score(
    X_test_tfidf,
    y_test
)

print("\n========== INDEPENDENT TEST ==========")

print(
    f"Test Accuracy    : {baseline_test_accuracy:.4f}"
)

print(
    "\nBaseline NLP classification completed successfully."
)

========== BASELINE NLP MODEL ==========
Model: Logistic Regression
CV strategy: StratifiedKFold
Number of folds: 5

========== CROSS-VALIDATION ==========
Fold scores: [0.73   0.7198 0.7044 0.7167 0.7228]
Mean CV Accuracy : 0.7188
CV Std           : 0.0084
CV Variance      : 0.0001

========== INDEPENDENT TEST ==========
Test Accuracy    : 0.6719

Baseline NLP classification completed successfully.


In [6]:
# =========================================================
# TASK 18 — CELL 6
# DETAILED CLASSIFICATION EVALUATION
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ---------------------------------------------------------
# Generate predictions
# ---------------------------------------------------------

y_test_pred = baseline_model.predict(
    X_test_tfidf
)

# ---------------------------------------------------------
# Calculate metrics
# ---------------------------------------------------------

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro",
    zero_division=0
)

# ---------------------------------------------------------
# Display overall metrics
# ---------------------------------------------------------

print("========== NLP EVALUATION ==========")

print(
    f"Accuracy          : {test_accuracy:.4f}"
)

print(
    f"Macro Precision   : {test_precision:.4f}"
)

print(
    f"Macro Recall      : {test_recall:.4f}"
)

print(
    f"Macro F1 Score    : {test_f1:.4f}"
)

# ---------------------------------------------------------
# Classification report
# ---------------------------------------------------------

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=target_names,
        digits=4,
        zero_division=0
    )
)

# ---------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_test_pred
)

print("\n========== CONFUSION MATRIX ==========")

print(
    "Shape:",
    cm.shape
)

print(cm)

print(
    "\nDetailed NLP evaluation completed successfully."
)

========== NLP EVALUATION ==========
Accuracy          : 0.6719
Macro Precision   : 0.6783
Macro Recall      : 0.6573
Macro F1 Score    : 0.6547

========== CLASSIFICATION REPORT ==========
                          precision    recall  f1-score   support

             alt.atheism     0.4749    0.4451    0.4595       319
           comp.graphics     0.5705    0.6658    0.6145       389
 comp.os.ms-windows.misc     0.6459    0.6066    0.6257       394
comp.sys.ibm.pc.hardware     0.6629    0.6020    0.6310       392
   comp.sys.mac.hardware     0.7003    0.6494    0.6739       385
          comp.windows.x     0.7943    0.7038    0.7463       395
            misc.forsale     0.7854    0.7974    0.7913       390
               rec.autos     0.7222    0.6894    0.7054       396
         rec.motorcycles     0.6893    0.7638    0.7247       398
      rec.sport.baseball     0.5145    0.8514    0.6414       397
        rec.sport.hockey     0.8932    0.8596    0.8761       399
               sc

In [7]:
# =========================================================
# TASK 18 — CELL 7
# ERROR ANALYSIS
# =========================================================

# ---------------------------------------------------------
# Identify incorrect predictions
# ---------------------------------------------------------

error_mask = y_test != y_test_pred

error_indices = np.where(error_mask)[0]

error_df = pd.DataFrame({
    "text": test_df.iloc[error_indices]["text"].values,
    "true_label": y_test[error_indices],
    "predicted_label": y_test_pred[error_indices]
})

# ---------------------------------------------------------
# Add class names
# ---------------------------------------------------------

error_df["true_class"] = error_df["true_label"].apply(
    lambda x: target_names[x]
)

error_df["predicted_class"] = error_df["predicted_label"].apply(
    lambda x: target_names[x]
)

# ---------------------------------------------------------
# Error count
# ---------------------------------------------------------

total_errors = len(error_df)

error_rate = total_errors / len(y_test)

print("========== ERROR ANALYSIS ==========")

print(
    "Total test samples:",
    len(y_test)
)

print(
    "Incorrect predictions:",
    total_errors
)

print(
    f"Error rate: {error_rate:.4f}"
)

# ---------------------------------------------------------
# Most common confusion pairs
# ---------------------------------------------------------

confusion_pairs = (
    error_df
    .groupby(
        ["true_class", "predicted_class"]
    )
    .size()
    .reset_index(
        name="error_count"
    )
    .sort_values(
        "error_count",
        ascending=False
    )
)

print(
    "\n========== MOST COMMON ERROR PAIRS =========="
)

print(
    confusion_pairs.head(15).to_string(
        index=False
    )
)

# ---------------------------------------------------------
# Show example errors
# ---------------------------------------------------------

print(
    "\n========== EXAMPLE MISCLASSIFICATIONS =========="
)

for i, row in error_df.head(10).iterrows():

    print("\n----------------------------------------")

    print(
        "True class     :",
        row["true_class"]
    )

    print(
        "Predicted class:",
        row["predicted_class"]
    )

    print(
        "Text preview   :",
        row["text"][:500].replace(
            "\n",
            " "
        )
    )

print(
    "\nError analysis completed successfully."
)

========== ERROR ANALYSIS ==========
Total test samples: 7532
Incorrect predictions: 2471
Error rate: 0.3281

========== MOST COMMON ERROR PAIRS ==========
              true_class          predicted_class  error_count
      talk.politics.misc       talk.politics.guns           96
      talk.religion.misc   soc.religion.christian           82
             alt.atheism   soc.religion.christian           67
          comp.windows.x            comp.graphics           46
      talk.religion.misc              alt.atheism           45
comp.sys.ibm.pc.hardware    comp.sys.mac.hardware           38
comp.sys.ibm.pc.hardware          sci.electronics           34
comp.sys.ibm.pc.hardware  comp.os.ms-windows.misc           33
 comp.os.ms-windows.misc comp.sys.ibm.pc.hardware           32
   comp.sys.mac.hardware comp.sys.ibm.pc.hardware           32
               rec.autos       rec.sport.baseball           31
        rec.sport.hockey       rec.sport.baseball           30
          comp.windows.x 

In [8]:
# =========================================================
# TASK 18 — CELL 8
# LINEAR SVM NLP MODEL
# =========================================================

from sklearn.svm import LinearSVC

# ---------------------------------------------------------
# Create Linear SVM
# ---------------------------------------------------------

svm_model = LinearSVC(
    random_state=RANDOM_STATE,
    C=1.0
)

print("========== LINEAR SVM NLP MODEL ==========")

print("Model: LinearSVC")
print("C:", 1.0)
print("CV strategy: StratifiedKFold")
print("Number of folds:", N_SPLITS)

# ---------------------------------------------------------
# Cross-validation
# ---------------------------------------------------------

svm_cv_scores = cross_val_score(
    svm_model,
    X_train_tfidf,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

# ---------------------------------------------------------
# CV statistics
# ---------------------------------------------------------

svm_cv_mean = svm_cv_scores.mean()
svm_cv_std = svm_cv_scores.std()
svm_cv_variance = svm_cv_scores.var()

print("\n========== CROSS-VALIDATION ==========")

print(
    "Fold scores:",
    np.round(svm_cv_scores, 4)
)

print(
    f"Mean CV Accuracy : {svm_cv_mean:.4f}"
)

print(
    f"CV Std           : {svm_cv_std:.4f}"
)

print(
    f"CV Variance      : {svm_cv_variance:.4f}"
)

# ---------------------------------------------------------
# Train SVM
# ---------------------------------------------------------

svm_model.fit(
    X_train_tfidf,
    y_train
)

# ---------------------------------------------------------
# Independent test evaluation
# ---------------------------------------------------------

svm_test_accuracy = svm_model.score(
    X_test_tfidf,
    y_test
)

print("\n========== INDEPENDENT TEST ==========")

print(
    f"Test Accuracy    : {svm_test_accuracy:.4f}"
)

# ---------------------------------------------------------
# Compare with baseline
# ---------------------------------------------------------

print("\n========== BASELINE COMPARISON ==========")

print(
    f"Logistic Regression CV : {cv_mean:.4f}"
)

print(
    f"Linear SVM CV          : {svm_cv_mean:.4f}"
)

print(
    f"Logistic Regression Test : "
    f"{baseline_test_accuracy:.4f}"
)

print(
    f"Linear SVM Test          : "
    f"{svm_test_accuracy:.4f}"
)

print(
    "\nLinear SVM evaluation completed successfully."
)

========== LINEAR SVM NLP MODEL ==========
Model: LinearSVC
C: 1.0
CV strategy: StratifiedKFold
Number of folds: 5

========== CROSS-VALIDATION ==========
Fold scores: [0.7534 0.7481 0.7362 0.734  0.7546]
Mean CV Accuracy : 0.7453
CV Std           : 0.0086
CV Variance      : 0.0001

========== INDEPENDENT TEST ==========
Test Accuracy    : 0.6796

========== BASELINE COMPARISON ==========
Logistic Regression CV : 0.7188
Linear SVM CV          : 0.7453
Logistic Regression Test : 0.6719
Linear SVM Test          : 0.6796

Linear SVM evaluation completed successfully.


In [9]:
# =========================================================
# TASK 18 — CELL 9
# MODEL COMPARISON
# =========================================================

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM"
    ],
    "Mean_CV_Accuracy": [
        cv_mean,
        svm_cv_mean
    ],
    "CV_Std": [
        cv_std,
        svm_cv_std
    ],
    "CV_Variance": [
        cv_variance,
        svm_cv_variance
    ],
    "Test_Accuracy": [
        baseline_test_accuracy,
        svm_test_accuracy
    ]
})

# ---------------------------------------------------------
# Calculate improvements
# ---------------------------------------------------------

baseline_cv = cv_mean
baseline_test = baseline_test_accuracy

model_comparison["CV_Gain_vs_Baseline"] = (
    model_comparison["Mean_CV_Accuracy"]
    - baseline_cv
)

model_comparison["Test_Gain_vs_Baseline"] = (
    model_comparison["Test_Accuracy"]
    - baseline_test
)

# ---------------------------------------------------------
# Rank models by CV performance
# ---------------------------------------------------------

model_comparison = model_comparison.sort_values(
    by="Mean_CV_Accuracy",
    ascending=False
).reset_index(drop=True)

model_comparison["CV_Rank"] = (
    model_comparison.index + 1
)

# ---------------------------------------------------------
# Display comparison
# ---------------------------------------------------------

print("========== NLP MODEL COMPARISON ==========")

print(
    model_comparison.round(4).to_string(
        index=False
    )
)

# ---------------------------------------------------------
# Select current best model
# ---------------------------------------------------------

best_model_name = model_comparison.iloc[0]["Model"]

best_cv_accuracy = model_comparison.iloc[0][
    "Mean_CV_Accuracy"
]

best_test_accuracy = model_comparison.iloc[0][
    "Test_Accuracy"
]

print("\n========== CURRENT BEST MODEL ==========")

print(
    "Selected model:",
    best_model_name
)

print(
    f"Mean CV accuracy: {best_cv_accuracy:.4f}"
)

print(
    f"Independent test accuracy: {best_test_accuracy:.4f}"
)

print(
    "\nModel comparison completed successfully."
)

========== NLP MODEL COMPARISON ==========
              Model  Mean_CV_Accuracy  CV_Std  CV_Variance  Test_Accuracy  CV_Gain_vs_Baseline  Test_Gain_vs_Baseline  CV_Rank
         Linear SVM            0.7453  0.0086       0.0001         0.6796               0.0265                 0.0077        1
Logistic Regression            0.7188  0.0084       0.0001         0.6719               0.0000                 0.0000        2

========== CURRENT BEST MODEL ==========
Selected model: Linear SVM
Mean CV accuracy: 0.7453
Independent test accuracy: 0.6796

Model comparison completed successfully.


In [10]:
# =========================================================
# TASK 18 — CELL 10
# PACKAGE NLP PIPELINE
# =========================================================

from pathlib import Path
import joblib
import json

# ---------------------------------------------------------
# Create directories
# ---------------------------------------------------------

ARTIFACTS_DIR = Path(
    "/home/akash/Projects/Altrodav/artifacts"
)

MODELS_DIR = Path(
    "/home/akash/Projects/Altrodav/models"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# Train final Linear SVM on complete training data
# ---------------------------------------------------------

final_nlp_model = LinearSVC(
    random_state=RANDOM_STATE,
    C=1.0
)

final_nlp_model.fit(
    X_train_tfidf,
    y_train
)

# ---------------------------------------------------------
# Save TF-IDF vectorizer
# ---------------------------------------------------------

vectorizer_path = (
    MODELS_DIR /
    "task18_tfidf_vectorizer.pkl"
)

joblib.dump(
    tfidf_vectorizer,
    vectorizer_path
)

# ---------------------------------------------------------
# Save Linear SVM model
# ---------------------------------------------------------

model_path = (
    MODELS_DIR /
    "task18_linear_svm_model.pkl"
)

joblib.dump(
    final_nlp_model,
    model_path
)

# ---------------------------------------------------------
# Save model comparison
# ---------------------------------------------------------

comparison_path = (
    ARTIFACTS_DIR /
    "task18_model_comparison.csv"
)

model_comparison.to_csv(
    comparison_path,
    index=False
)

# ---------------------------------------------------------
# Save metadata
# ---------------------------------------------------------

metadata = {
    "task": "Task 18 — Natural Language Processing",
    "random_state": RANDOM_STATE,
    "cv_strategy": "StratifiedKFold",
    "cv_folds": N_SPLITS,
    "vectorizer": "TF-IDF",
    "ngram_range": [1, 2],
    "max_features": 50000,
    "selected_model": "LinearSVC",
    "C": 1.0,
    "mean_cv_accuracy": float(svm_cv_mean),
    "cv_std": float(svm_cv_std),
    "cv_variance": float(svm_cv_variance),
    "test_accuracy": float(svm_test_accuracy),
    "number_of_classes": len(target_names),
    "training_samples": len(y_train),
    "test_samples": len(y_test)
}

metadata_path = (
    ARTIFACTS_DIR /
    "task18_nlp_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

# ---------------------------------------------------------
# Display saved artifacts
# ---------------------------------------------------------

print("========== NLP PIPELINE ARTIFACTS ==========")

print(
    "TF-IDF vectorizer:",
    vectorizer_path
)

print(
    "Linear SVM model:",
    model_path
)

print(
    "Model comparison:",
    comparison_path
)

print(
    "NLP metadata:",
    metadata_path
)

print(
    "\nAll NLP pipeline artifacts saved successfully."
)

========== NLP PIPELINE ARTIFACTS ==========
TF-IDF vectorizer: /home/akash/Projects/Altrodav/models/task18_tfidf_vectorizer.pkl
Linear SVM model: /home/akash/Projects/Altrodav/models/task18_linear_svm_model.pkl
Model comparison: /home/akash/Projects/Altrodav/artifacts/task18_model_comparison.csv
NLP metadata: /home/akash/Projects/Altrodav/artifacts/task18_nlp_metadata.json

All NLP pipeline artifacts saved successfully.


In [11]:
# =========================================================
# TASK 18 — CELL 11
# END-TO-END NLP PIPELINE VERIFICATION
# =========================================================

# ---------------------------------------------------------
# Load saved artifacts
# ---------------------------------------------------------

loaded_vectorizer = joblib.load(
    "/home/akash/Projects/Altrodav/models/task18_tfidf_vectorizer.pkl"
)

loaded_model = joblib.load(
    "/home/akash/Projects/Altrodav/models/task18_linear_svm_model.pkl"
)

# ---------------------------------------------------------
# New text examples
# ---------------------------------------------------------

new_texts = [
    """
    I am looking for information about graphics cards,
    display resolution, rendering and computer graphics.
    """,

    """
    The car has a powerful engine, excellent acceleration,
    and good fuel economy. I want to compare different models.
    """,

    """
    NASA launched a spacecraft to study planets, stars,
    galaxies and the future of space exploration.
    """,

    """
    I am interested in computer hardware, processors,
    memory, motherboard and PCI expansion cards.
    """
]

# ---------------------------------------------------------
# Transform new text using the SAVED vectorizer
# ---------------------------------------------------------

new_text_tfidf = loaded_vectorizer.transform(
    new_texts
)

# ---------------------------------------------------------
# Predict using the SAVED model
# ---------------------------------------------------------

new_predictions = loaded_model.predict(
    new_text_tfidf
)

# ---------------------------------------------------------
# Display predictions
# ---------------------------------------------------------

print("========== END-TO-END NLP VERIFICATION ==========")

for i, (text, prediction) in enumerate(
    zip(new_texts, new_predictions),
    start=1
):

    print("\n----------------------------------------")

    print(
        f"Example {i}"
    )

    print(
        "Text:",
        " ".join(text.split())
    )

    print(
        "Predicted class:",
        target_names[prediction]
    )

# ---------------------------------------------------------
# Shape verification
# ---------------------------------------------------------

print("\n========== PIPELINE CHECK ==========")

print(
    "Loaded TF-IDF feature count:",
    len(loaded_vectorizer.vocabulary_)
)

print(
    "Transformed new text shape:",
    new_text_tfidf.shape
)

print(
    "Number of predictions:",
    len(new_predictions)
)

print(
    "\nSaved vectorizer loading : PASS"
)

print(
    "Saved model loading      : PASS"
)

print(
    "New text transformation  : PASS"
)

print(
    "New text prediction      : PASS"
)

print(
    "\nEnd-to-end NLP pipeline verification completed successfully."
)

========== END-TO-END NLP VERIFICATION ==========

----------------------------------------
Example 1
Text: I am looking for information about graphics cards, display resolution, rendering and computer graphics.
Predicted class: comp.graphics

----------------------------------------
Example 2
Text: The car has a powerful engine, excellent acceleration, and good fuel economy. I want to compare different models.
Predicted class: rec.autos

----------------------------------------
Example 3
Text: NASA launched a spacecraft to study planets, stars, galaxies and the future of space exploration.
Predicted class: sci.space

----------------------------------------
Example 4
Text: I am interested in computer hardware, processors, memory, motherboard and PCI expansion cards.
Predicted class: comp.sys.ibm.pc.hardware

========== PIPELINE CHECK ==========
Loaded TF-IDF feature count: 50000
Transformed new text shape: (4, 50000)
Number of predictions: 4

Saved vectorizer loading : PASS
Saved mode

In [12]:
# =========================================================
# TASK 18 — FINAL VERIFICATION
# =========================================================

from pathlib import Path
import joblib
import json
import pandas as pd

print("=" * 65)
print("             TASK 18 — FINAL VERIFICATION")
print("=" * 65)

# ---------------------------------------------------------
# Artifact paths
# ---------------------------------------------------------

vectorizer_path = Path(
    "/home/akash/Projects/Altrodav/models/task18_tfidf_vectorizer.pkl"
)

model_path = Path(
    "/home/akash/Projects/Altrodav/models/task18_linear_svm_model.pkl"
)

comparison_path = Path(
    "/home/akash/Projects/Altrodav/artifacts/task18_model_comparison.csv"
)

metadata_path = Path(
    "/home/akash/Projects/Altrodav/artifacts/task18_nlp_metadata.json"
)

# ---------------------------------------------------------
# Check files exist
# ---------------------------------------------------------

print("\n========== ARTIFACT CHECK ==========")

print(
    "TF-IDF vectorizer:",
    "PASS" if vectorizer_path.exists() else "FAIL"
)

print(
    "Linear SVM model:",
    "PASS" if model_path.exists() else "FAIL"
)

print(
    "Model comparison:",
    "PASS" if comparison_path.exists() else "FAIL"
)

print(
    "NLP metadata:",
    "PASS" if metadata_path.exists() else "FAIL"
)

# ---------------------------------------------------------
# Load artifacts
# ---------------------------------------------------------

loaded_vectorizer = joblib.load(
    vectorizer_path
)

loaded_model = joblib.load(
    model_path
)

loaded_comparison = pd.read_csv(
    comparison_path
)

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as f:

    loaded_metadata = json.load(f)

print("\n========== ARTIFACT LOADING ==========")

print("Vectorizer loading : PASS")
print("Model loading      : PASS")
print("Comparison loading : PASS")
print("Metadata loading   : PASS")

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

print("\n========== NLP CONFIGURATION ==========")

print(
    "Vectorizer        :",
    loaded_metadata["vectorizer"]
)

print(
    "N-gram range      :",
    loaded_metadata["ngram_range"]
)

print(
    "Maximum features  :",
    loaded_metadata["max_features"]
)

print(
    "Selected model    :",
    loaded_metadata["selected_model"]
)

print(
    "Number of classes :",
    loaded_metadata["number_of_classes"]
)

print(
    "Training samples  :",
    loaded_metadata["training_samples"]
)

print(
    "Test samples      :",
    loaded_metadata["test_samples"]
)

# ---------------------------------------------------------
# Performance
# ---------------------------------------------------------

print("\n========== FINAL NLP PERFORMANCE ==========")

print(
    f"Mean CV accuracy : "
    f"{loaded_metadata['mean_cv_accuracy']:.4f}"
)

print(
    f"CV standard deviation : "
    f"{loaded_metadata['cv_std']:.4f}"
)

print(
    f"CV variance : "
    f"{loaded_metadata['cv_variance']:.4f}"
)

print(
    f"Independent test accuracy : "
    f"{loaded_metadata['test_accuracy']:.4f}"
)

# ---------------------------------------------------------
# Final conclusion
# ---------------------------------------------------------

print("\n========== FINAL CONCLUSION ==========")

print(
    "Linear SVM was selected as the best NLP classifier "
    "based on cross-validation performance."
)

print(
    "The NLP pipeline successfully converts cleaned text "
    "into TF-IDF features and generates class predictions."
)

print(
    "The packaged vectorizer and model were successfully "
    "loaded and verified on new text."
)

print("\n" + "=" * 65)
print("       TASK 18 COMPLETED SUCCESSFULLY!")
print("=" * 65)

             TASK 18 — FINAL VERIFICATION

========== ARTIFACT CHECK ==========
TF-IDF vectorizer: PASS
Linear SVM model: PASS
Model comparison: PASS
NLP metadata: PASS

========== ARTIFACT LOADING ==========
Vectorizer loading : PASS
Model loading      : PASS
Comparison loading : PASS
Metadata loading   : PASS

========== NLP CONFIGURATION ==========
Vectorizer        : TF-IDF
N-gram range      : [1, 2]
Maximum features  : 50000
Selected model    : LinearSVC
Number of classes : 20
Training samples  : 11314
Test samples      : 7532

========== FINAL NLP PERFORMANCE ==========
Mean CV accuracy : 0.7453
CV standard deviation : 0.0086
CV variance : 0.0001
Independent test accuracy : 0.6796

========== FINAL CONCLUSION ==========
Linear SVM was selected as the best NLP classifier based on cross-validation performance.
The NLP pipeline successfully converts cleaned text into TF-IDF features and generates class predictions.
The packaged vectorizer and model were successfully loaded and verif